In [1]:
# ============================================================
# OSM FACILITIES BY LAD AND MSOA — ENGLAND
# Downloads 8 facility types from OpenStreetMap via Overpass API
# Aggregates to LAD and MSOA level
# Outputs: raw counts + per-100k population
# ============================================================

import requests
import pandas as pd
import geopandas as gpd
import numpy as np
import time
from pathlib import Path

OUT_DIR = Path(r"G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory\output\Diane_final_version")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Boundary files (already in your project) ─────────────────
LAD_BOUNDARY  = (
    r"G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory"
    r"\Data\raw_data\boundaries"
    r"\UK_Local_Authority_Districts_December_2023_Boundaries_UK_BGC_2537431731774104276.GeoJSON"
)

# MSOA boundary — download from ONS if you don't have it:
# https://geoportal.statistics.gov.uk → search "MSOA Dec 2021 EW BGC"
MSOA_BOUNDARY = (
    r"G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory"
    r"\Data\raw_data\boundaries"
    r"\MSOA_Dec_2021_EW_BGC.GeoJSON"
)

# ONS population estimates by LAD (for per-100k calculation)
# Download from: https://www.ons.gov.uk/peoplepopulationandcommunity/
#                populationandmigration/populationestimates
# or use your existing df if it has a population column
POP_CSV = (
    r"G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory"
    r"\Data\raw_data\lad_population_estimates.csv"
)

# ============================================================
# 1. FACILITY DEFINITIONS
# OSM tag combinations for each facility type
# ============================================================
FACILITIES = {
    "universities": {
        "tags": [
            '["amenity"="university"]',
            '["amenity"="college"]["isced:level"~"6|7|8"]',
        ],
        "label": "Universities and HE institutions",
    },
    "hospitals": {
        "tags": [
            '["amenity"="hospital"]',
            '["amenity"="clinic"]["healthcare"="hospital"]',
        ],
        "label": "Hospitals and NHS facilities",
    },
    "schools": {
        "tags": [
            '["amenity"="school"]',
        ],
        "label": "Schools (primary and secondary)",
    },
    "gp_surgeries": {
        "tags": [
            '["amenity"="doctors"]',
            '["healthcare"="doctor"]',
            '["amenity"="clinic"]',
        ],
        "label": "GP surgeries and clinics",
    },
    "residential": {
        "tags": [
            '["building"="residential"]',
            '["building"="house"]',
            '["building"="apartments"]',
            '["building"="terrace"]',
            '["building"="detached"]',
            '["building"="semidetached_house"]',
        ],
        "label": "Residential buildings",
    },
    "industrial": {
        "tags": [
            '["landuse"="industrial"]',
            '["building"="industrial"]',
            '["building"="warehouse"]',
            '["building"="factory"]',
        ],
        "label": "Industrial and manufacturing sites",
    },
    "research_centres": {
        "tags": [
            '["amenity"="research_institute"]',
            '["office"="research"]',
            '["building"="laboratory"]',
            '["landuse"="research"]',
        ],
        "label": "Research centres and science parks",
    },
    "fe_colleges": {
        "tags": [
            '["amenity"="college"]',
            '["building"="college"]',
        ],
        "label": "Further education colleges",
    },
}

# ============================================================
# 2. OVERPASS QUERY FUNCTION
# ============================================================
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

def build_query(tags, area="England", timeout=180):
    """
    Builds an Overpass QL query for nodes, ways and relations
    matching any of the given tags within England.
    Returns lat/lon center point for each feature.
    """
    tag_blocks = ""
    for tag in tags:
        tag_blocks += f"""
      node{tag}(area.england);
      way{tag}(area.england);
      relation{tag}(area.england);"""

    query = f"""
    [out:json][timeout:{timeout}];
    area["name"="England"]["boundary"="administrative"]->.england;
    (
      {tag_blocks}
    );
    out center;
    """
    return query


def fetch_osm(facility_name, tags, retries=3, pause=10):
    """
    Fetches OSM data for a facility type.
    Returns a DataFrame with lat, lon, osm_type, osm_id.
    """
    print(f"  Fetching: {facility_name}...", end=" ")
    query = build_query(tags)

    for attempt in range(retries):
        try:
            resp = requests.post(
                OVERPASS_URL,
                data={"data": query},
                timeout=300,
            )
            resp.raise_for_status()
            data = resp.json()

            records = []
            for el in data.get("elements", []):
                # nodes have lat/lon directly
                # ways/relations have a center
                lat = el.get("lat") or (
                    el.get("center", {}).get("lat"))
                lon = el.get("lon") or (
                    el.get("center", {}).get("lon"))
                if lat and lon:
                    records.append({
                        "osm_id":   el["id"],
                        "osm_type": el["type"],
                        "lat":      lat,
                        "lon":      lon,
                        "facility": facility_name,
                    })

            df = pd.DataFrame(records)
            print(f"{len(df):,} features found")
            return df

        except Exception as e:
            print(f"\n    Attempt {attempt+1} failed: {e}")
            if attempt < retries - 1:
                print(f"    Retrying in {pause}s...")
                time.sleep(pause)

    print(f"    FAILED after {retries} attempts")
    return pd.DataFrame()


# ============================================================
# 3. DOWNLOAD ALL FACILITIES
# ============================================================
print("=" * 60)
print("DOWNLOADING OSM FACILITIES FOR ENGLAND")
print("=" * 60)

all_dfs = []
for name, cfg in FACILITIES.items():
    df = fetch_osm(name, cfg["tags"])
    if not df.empty:
        all_dfs.append(df)
    time.sleep(5)   # polite pause between queries

if not all_dfs:
    raise RuntimeError("No data downloaded — check internet connection")

gdf_points = gpd.GeoDataFrame(
    pd.concat(all_dfs, ignore_index=True),
    geometry=gpd.points_from_xy(
        pd.concat(all_dfs, ignore_index=True)["lon"],
        pd.concat(all_dfs, ignore_index=True)["lat"],
    ),
    crs="EPSG:4326",
)

print(f"\nTotal features downloaded: {len(gdf_points):,}")
print(gdf_points["facility"].value_counts().to_string())

# Save raw points
gdf_points.to_file(
    OUT_DIR / "osm_raw_points_fv.gpkg",
    driver="GPKG")
print(f"\nRaw points saved → {OUT_DIR}/osm_raw_points_fv.gpkg")


# ============================================================
# 4. SPATIAL JOIN TO LAD
# ============================================================
print("\nLoading LAD boundaries...")
lad = gpd.read_file(LAD_BOUNDARY)
lad = lad[lad["LAD23CD"].str.startswith("E")].copy()
lad = lad.to_crs("EPSG:4326")
print(f"  {len(lad)} English LADs loaded")

print("Spatial join: points → LAD...")
joined_lad = gpd.sjoin(
    gdf_points,
    lad[["LAD23CD", "LAD23NM", "geometry"]],
    how="left",
    predicate="within",
)

# Pivot: one row per LAD, one column per facility type
lad_counts = (
    joined_lad
    .groupby(["LAD23CD", "LAD23NM", "facility"])
    .size()
    .reset_index(name="count")
    .pivot(index=["LAD23CD", "LAD23NM"],
           columns="facility",
           values="count")
    .fillna(0)
    .astype(int)
    .reset_index()
)

# Rename columns with clean prefix
lad_counts.columns.name = None
rename_lad = {f: f"n_{f}_lad" for f in FACILITIES.keys()}
lad_counts = lad_counts.rename(columns=rename_lad)
print(f"  LAD counts shape: {lad_counts.shape}")


# ============================================================
# 5. SPATIAL JOIN TO MSOA
# ============================================================
print("\nLoading MSOA boundaries...")
try:
    msoa = gpd.read_file(MSOA_BOUNDARY)
    msoa = msoa[msoa.iloc[:, 0].str.startswith("E")].copy()
    msoa = msoa.to_crs("EPSG:4326")
    msoa_code_col = msoa.columns[0]
    msoa_name_col = msoa.columns[1]
    print(f"  {len(msoa)} English MSOAs loaded")

    print("Spatial join: points → MSOA...")
    joined_msoa = gpd.sjoin(
        gdf_points,
        msoa[[msoa_code_col, msoa_name_col, "geometry"]],
        how="left",
        predicate="within",
    )

    msoa_counts = (
        joined_msoa
        .groupby([msoa_code_col, msoa_name_col, "facility"])
        .size()
        .reset_index(name="count")
        .pivot(index=[msoa_code_col, msoa_name_col],
               columns="facility",
               values="count")
        .fillna(0)
        .astype(int)
        .reset_index()
    )
    msoa_counts.columns.name = None
    rename_msoa = {f: f"n_{f}_msoa" for f in FACILITIES.keys()}
    msoa_counts = msoa_counts.rename(columns=rename_msoa)
    print(f"  MSOA counts shape: {msoa_counts.shape}")
    msoa_available = True

except FileNotFoundError:
    print("  MSOA boundary file not found — skipping MSOA level")
    print("  Download from: https://geoportal.statistics.gov.uk")
    msoa_available = False


# ============================================================
# 6. ADD PER-100K POPULATION
# ============================================================
print("\nAdding per-100k population rates...")

try:
    pop = pd.read_csv(POP_CSV)
    # Expects columns: geography_code, population
    # Adjust column names if different in your file
    pop = pop[["geography_code", "population"]].rename(
        columns={"geography_code": "LAD23CD"})

    lad_counts = lad_counts.merge(pop, on="LAD23CD", how="left")

    count_cols = [c for c in lad_counts.columns
                  if c.startswith("n_") and c.endswith("_lad")]

    for col in count_cols:
        rate_col = col.replace("n_", "rate_").replace(
            "_lad", "_per100k_lad")
        lad_counts[rate_col] = (
            lad_counts[col] / lad_counts["population"] * 100_000
        ).round(2)

    print(f"  Per-100k columns added for LAD")
    pop_available = True

except FileNotFoundError:
    print("  Population file not found — skipping per-100k rates")
    print("  Expected path:", POP_CSV)
    pop_available = False


# ============================================================
# 7. EXPORT TO EXCEL AND CSV
# ============================================================
print("\nExporting results...")

# CSV exports
lad_counts.to_csv(
    OUT_DIR / "facilities_by_lad_fv.csv", index=False)
print(f"  LAD CSV → {OUT_DIR}/facilities_by_lad_fv.csv")

if msoa_available:
    msoa_counts.to_csv(
        OUT_DIR / "facilities_by_msoa_fv.csv", index=False)
    print(f"  MSOA CSV → {OUT_DIR}/facilities_by_msoa_fv.csv")

# Excel export — one sheet per geographic level
xl_path = OUT_DIR / "facilities_osm_fv.xlsx"
with pd.ExcelWriter(xl_path, engine="openpyxl") as writer:

    # Sheet 1 — LAD raw counts
    lad_counts.to_excel(
        writer, sheet_name="LAD_counts", index=False)

    # Sheet 2 — LAD per 100k (if population available)
    if pop_available:
        rate_cols = ["LAD23CD", "LAD23NM"] + [
            c for c in lad_counts.columns
            if "per100k" in c]
        lad_counts[rate_cols].to_excel(
            writer, sheet_name="LAD_per100k", index=False)

    # Sheet 3 — MSOA counts
    if msoa_available:
        msoa_counts.to_excel(
            writer, sheet_name="MSOA_counts", index=False)

    # Sheet 4 — Summary: total by facility type
    summary_rows = []
    for fac, cfg in FACILITIES.items():
        col = f"n_{fac}_lad"
        if col in lad_counts.columns:
            summary_rows.append({
                "facility":    fac,
                "label":       cfg["label"],
                "total_count": int(lad_counts[col].sum()),
                "lad_median":  lad_counts[col].median(),
                "lad_max":     int(lad_counts[col].max()),
                "lad_zero_pct": round(
                    (lad_counts[col] == 0).mean() * 100, 1),
            })
    pd.DataFrame(summary_rows).to_excel(
        writer, sheet_name="Summary", index=False)

print(f"  Excel → {xl_path}")

# ============================================================
# 8. QUICK DIAGNOSTIC PRINT
# ============================================================
print("\n" + "=" * 60)
print("SUMMARY — FACILITIES DOWNLOADED")
print("=" * 60)
for fac, cfg in FACILITIES.items():
    col = f"n_{fac}_lad"
    if col in lad_counts.columns:
        total = int(lad_counts[col].sum())
        median = lad_counts[col].median()
        zeros = int((lad_counts[col] == 0).sum())
        print(f"  {fac:25s} total={total:6,} | "
              f"median/LAD={median:.1f} | "
              f"LADs with zero={zeros}")

print(f"\n✅ All done → {OUT_DIR}")

DOWNLOADING OSM FACILITIES FOR ENGLAND
  Fetching: universities... 935 features found
  Fetching: hospitals... 
    Attempt 1 failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    Retrying in 10s...

    Attempt 2 failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    Retrying in 10s...
1,486 features found
  Fetching: schools... 
    Attempt 1 failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    Retrying in 10s...
25,552 features found
  Fetching: gp_surgeries... 
    Attempt 1 failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    Retrying in 10s...

    Attempt 2 failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    Retrying in 10s...

    Attempt 3 failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    FAILED after 3 attempts
  Fetching: resi

c:\Users\diane\anaconda3\envs\pp422\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()
c:\Users\diane\anaconda3\envs\pp422\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Cannot find tms_NZTM2000.json (GDAL_DATA is not defined)
  ogr_write(



Raw points saved → G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory\output\Diane_final_version/osm_raw_points_fv.gpkg

Loading LAD boundaries...
  296 English LADs loaded
Spatial join: points → LAD...
  LAD counts shape: (296, 6)

Loading MSOA boundaries...


DataSourceError: G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory\Data\raw_data\boundaries\MSOA_Dec_2021_EW_BGC.GeoJSON: No such file or directory

In [8]:
# ============================================================
# OSM FACILITIES BY LAD AND MSOA — ENGLAND
# Downloads 8 facility types from OpenStreetMap via Overpass API
# Aggregates to LAD and MSOA level
# Outputs: raw counts + per-100k population
# ============================================================

import requests
import pandas as pd
import geopandas as gpd
import numpy as np
import time
from pathlib import Path

OUT_DIR = Path(r"G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory\output\Diane_final_version")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Boundary files (already in your project) ─────────────────
LAD_BOUNDARY  = (
    r"G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory"
    r"\Data\raw_data\boundaries"
    r"\UK_Local_Authority_Districts_December_2023_Boundaries_UK_BGC_2537431731774104276.GeoJSON"
)

MSOA_BOUNDARY = (
    r"G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory"
    r"\Data\raw_data\boundaries"
    r"\Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3_-4477917303172606123.GeoJSON"
)

# ONS population estimates by LAD (for per-100k calculation)
# Download from: https://www.ons.gov.uk/peoplepopulationandcommunity/
#                populationandmigration/populationestimates
# or use your existing df if it has a population column
POP_CSV = (
    r"G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory"
    r"\Data\raw_data\lad_population_estimates.csv"
)

# ============================================================
# 1. FACILITY DEFINITIONS
# OSM tag combinations for each facility type
# ============================================================
FACILITIES = {
    "universities": {
        "tags": [
            '["amenity"="university"]',
            '["amenity"="college"]["isced:level"~"6|7|8"]',
        ],
        "label": "Universities and HE institutions",
    },
    "hospitals": {
        "tags": [
            '["amenity"="hospital"]',
            '["amenity"="clinic"]["healthcare"="hospital"]',
        ],
        "label": "Hospitals and NHS facilities",
    },
    "schools": {
        "tags": [
            '["amenity"="school"]',
        ],
        "label": "Schools (primary and secondary)",
    },
    "gp_surgeries": {
        "tags": [
            '["amenity"="doctors"]',
            '["healthcare"="doctor"]',
            '["amenity"="clinic"]',
        ],
        "label": "GP surgeries and clinics",
    },
    "residential": {
        "tags": [
            '["building"="residential"]',
            '["building"="house"]',
            '["building"="apartments"]',
            '["building"="terrace"]',
            '["building"="detached"]',
            '["building"="semidetached_house"]',
        ],
        "label": "Residential buildings",
    },
    "industrial": {
        "tags": [
            '["landuse"="industrial"]',
            '["building"="industrial"]',
            '["building"="warehouse"]',
            '["building"="factory"]',
        ],
        "label": "Industrial and manufacturing sites",
    },
    "research_centres": {
        "tags": [
            '["amenity"="research_institute"]',
            '["office"="research"]',
            '["building"="laboratory"]',
            '["landuse"="research"]',
        ],
        "label": "Research centres and science parks",
    },
    "fe_colleges": {
        "tags": [
            '["amenity"="college"]',
            '["building"="college"]',
        ],
        "label": "Further education colleges",
    },
}

# ============================================================
# 2. OVERPASS QUERY FUNCTION
# ============================================================
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

def build_query(tags, area="England", timeout=180):
    """
    Builds an Overpass QL query for nodes, ways and relations
    matching any of the given tags within England.
    Returns lat/lon center point for each feature.
    """
    tag_blocks = ""
    for tag in tags:
        tag_blocks += f"""
      node{tag}(area.england);
      way{tag}(area.england);
      relation{tag}(area.england);"""

    query = f"""
    [out:json][timeout:{timeout}];
    area["name"="England"]["boundary"="administrative"]->.england;
    (
      {tag_blocks}
    );
    out center;
    """
    return query


def fetch_osm(facility_name, tags, retries=3, pause=10):
    """
    Fetches OSM data for a facility type.
    Returns a DataFrame with lat, lon, osm_type, osm_id.
    """
    print(f"  Fetching: {facility_name}...", end=" ")
    query = build_query(tags)

    for attempt in range(retries):
        try:
            resp = requests.post(
                OVERPASS_URL,
                data={"data": query},
                timeout=300,
            )
            resp.raise_for_status()
            data = resp.json()

            records = []
            for el in data.get("elements", []):
                # nodes have lat/lon directly
                # ways/relations have a center
                lat = el.get("lat") or (
                    el.get("center", {}).get("lat"))
                lon = el.get("lon") or (
                    el.get("center", {}).get("lon"))
                if lat and lon:
                    records.append({
                        "osm_id":   el["id"],
                        "osm_type": el["type"],
                        "lat":      lat,
                        "lon":      lon,
                        "facility": facility_name,
                    })

            df = pd.DataFrame(records)
            print(f"{len(df):,} features found")
            return df

        except Exception as e:
            print(f"\n    Attempt {attempt+1} failed: {e}")
            if attempt < retries - 1:
                print(f"    Retrying in {pause}s...")
                time.sleep(pause)

    print(f"    FAILED after {retries} attempts")
    return pd.DataFrame()


# ============================================================
# 3. DOWNLOAD ALL FACILITIES
# ============================================================
print("=" * 60)
print("DOWNLOADING OSM FACILITIES FOR ENGLAND")
print("=" * 60)

all_dfs = []
for name, cfg in FACILITIES.items():
    df = fetch_osm(name, cfg["tags"])
    if not df.empty:
        all_dfs.append(df)
    time.sleep(5)   # polite pause between queries

if not all_dfs:
    raise RuntimeError("No data downloaded — check internet connection")

gdf_points = gpd.GeoDataFrame(
    pd.concat(all_dfs, ignore_index=True),
    geometry=gpd.points_from_xy(
        pd.concat(all_dfs, ignore_index=True)["lon"],
        pd.concat(all_dfs, ignore_index=True)["lat"],
    ),
    crs="EPSG:4326",
)

print(f"\nTotal features downloaded: {len(gdf_points):,}")
print(gdf_points["facility"].value_counts().to_string())

# Save raw points
gdf_points.to_file(
    OUT_DIR / "osm_raw_points_fv.gpkg",
    driver="GPKG")
print(f"\nRaw points saved → {OUT_DIR}/osm_raw_points_fv.gpkg")


# ============================================================
# 4. SPATIAL JOIN TO LAD
# ============================================================
print("\nLoading LAD boundaries...")
lad = gpd.read_file(LAD_BOUNDARY)
lad = lad[lad["LAD23CD"].str.startswith("E")].copy()
lad = lad.to_crs("EPSG:4326")
print(f"  {len(lad)} English LADs loaded")

print("Spatial join: points → LAD...")
joined_lad = gpd.sjoin(
    gdf_points,
    lad[["LAD23CD", "LAD23NM", "geometry"]],
    how="left",
    predicate="within",
)

# Pivot: one row per LAD, one column per facility type
lad_counts = (
    joined_lad
    .groupby(["LAD23CD", "LAD23NM", "facility"])
    .size()
    .reset_index(name="count")
    .pivot(index=["LAD23CD", "LAD23NM"],
           columns="facility",
           values="count")
    .fillna(0)
    .astype(int)
    .reset_index()
)

# Rename columns with clean prefix
lad_counts.columns.name = None
rename_lad = {f: f"n_{f}_lad" for f in FACILITIES.keys()}
lad_counts = lad_counts.rename(columns=rename_lad)
print(f"  LAD counts shape: {lad_counts.shape}")


# ============================================================
# 5. SPATIAL JOIN TO MSOA
# ============================================================
print("\nLoading MSOA boundaries...")
try:
    msoa = gpd.read_file(MSOA_BOUNDARY)
    msoa = msoa[msoa["MSOA21CD"].str.startswith("E")].copy()
    msoa = msoa.to_crs("EPSG:4326")
    msoa_code_col = "MSOA21CD"
    msoa_name_col = "MSOA21NM"
    print(f"  {len(msoa)} English MSOAs loaded")

    print("Spatial join: points → MSOA...")
    joined_msoa = gpd.sjoin(
        gdf_points,
        msoa[[msoa_code_col, msoa_name_col, "geometry"]],
        how="left",
        predicate="within",
    )

    msoa_counts = (
        joined_msoa
        .groupby([msoa_code_col, msoa_name_col, "facility"])
        .size()
        .reset_index(name="count")
        .pivot(index=[msoa_code_col, msoa_name_col],
               columns="facility",
               values="count")
        .fillna(0)
        .astype(int)
        .reset_index()
    )
    msoa_counts.columns.name = None
    rename_msoa = {f: f"n_{f}_msoa" for f in FACILITIES.keys()}
    msoa_counts = msoa_counts.rename(columns=rename_msoa)
    print(f"  MSOA counts shape: {msoa_counts.shape}")
    msoa_available = True

except FileNotFoundError:
    print("  MSOA boundary file not found — skipping MSOA level")
    print("  Download from: https://geoportal.statistics.gov.uk")
    msoa_available = False


# ============================================================
# 6. ADD PER-100K POPULATION
# ============================================================
print("\nAdding per-100k population rates...")

try:
    pop = pd.read_csv(POP_CSV)
    # Expects columns: geography_code, population
    # Adjust column names if different in your file
    pop = pop[["geography_code", "population"]].rename(
        columns={"geography_code": "LAD23CD"})

    lad_counts = lad_counts.merge(pop, on="LAD23CD", how="left")

    count_cols = [c for c in lad_counts.columns
                  if c.startswith("n_") and c.endswith("_lad")]

    for col in count_cols:
        rate_col = col.replace("n_", "rate_").replace(
            "_lad", "_per100k_lad")
        lad_counts[rate_col] = (
            lad_counts[col] / lad_counts["population"] * 100_000
        ).round(2)

    print(f"  Per-100k columns added for LAD")
    pop_available = True

except FileNotFoundError:
    print("  Population file not found — skipping per-100k rates")
    print("  Expected path:", POP_CSV)
    pop_available = False


# ============================================================
# 7. EXPORT TO EXCEL AND CSV
# ============================================================
print("\nExporting results...")

# CSV exports
lad_counts.to_csv(
    OUT_DIR / "facilities_by_lad_fv.csv", index=False)
print(f"  LAD CSV → {OUT_DIR}/facilities_by_lad_fv.csv")

if msoa_available:
    msoa_counts.to_csv(
        OUT_DIR / "facilities_by_msoa_fv.csv", index=False)
    print(f"  MSOA CSV → {OUT_DIR}/facilities_by_msoa_fv.csv")

# Excel export — one sheet per geographic level
xl_path = OUT_DIR / "facilities_osm_fv.xlsx"
with pd.ExcelWriter(xl_path, engine="openpyxl") as writer:

    # Sheet 1 — LAD raw counts
    lad_counts.to_excel(
        writer, sheet_name="LAD_counts", index=False)

    # Sheet 2 — LAD per 100k (if population available)
    if pop_available:
        rate_cols = ["LAD23CD", "LAD23NM"] + [
            c for c in lad_counts.columns
            if "per100k" in c]
        lad_counts[rate_cols].to_excel(
            writer, sheet_name="LAD_per100k", index=False)

    # Sheet 3 — MSOA counts
    if msoa_available:
        msoa_counts.to_excel(
            writer, sheet_name="MSOA_counts", index=False)

    # Sheet 4 — Summary: total by facility type
    summary_rows = []
    for fac, cfg in FACILITIES.items():
        col = f"n_{fac}_lad"
        if col in lad_counts.columns:
            summary_rows.append({
                "facility":    fac,
                "label":       cfg["label"],
                "total_count": int(lad_counts[col].sum()),
                "lad_median":  lad_counts[col].median(),
                "lad_max":     int(lad_counts[col].max()),
                "lad_zero_pct": round(
                    (lad_counts[col] == 0).mean() * 100, 1),
            })
    pd.DataFrame(summary_rows).to_excel(
        writer, sheet_name="Summary", index=False)

print(f"  Excel → {xl_path}")

# ============================================================
# 8. QUICK DIAGNOSTIC PRINT
# ============================================================
print("\n" + "=" * 60)
print("SUMMARY — FACILITIES DOWNLOADED")
print("=" * 60)
for fac, cfg in FACILITIES.items():
    col = f"n_{fac}_lad"
    if col in lad_counts.columns:
        total = int(lad_counts[col].sum())
        median = lad_counts[col].median()
        zeros = int((lad_counts[col] == 0).sum())
        print(f"  {fac:25s} total={total:6,} | "
              f"median/LAD={median:.1f} | "
              f"LADs with zero={zeros}")

print(f"\n✅ All done → {OUT_DIR}")

DOWNLOADING OSM FACILITIES FOR ENGLAND
  Fetching: universities... 935 features found
  Fetching: hospitals... 
    Attempt 1 failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    Retrying in 10s...
1,486 features found
  Fetching: schools... 
    Attempt 1 failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    Retrying in 10s...
25,552 features found
  Fetching: gp_surgeries... 
    Attempt 1 failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    Retrying in 10s...
8,267 features found
  Fetching: residential... 
    Attempt 1 failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    Retrying in 10s...

    Attempt 2 failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    Retrying in 10s...

    Attempt 3 failed: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpre

In [9]:
import geopandas as gpd

lad = gpd.read_file(
    r"G:\Mi unidad\Master_LSE\Git-hub\Economic_Observatory"
    r"\Data\raw_data\boundaries"
    r"\UK_Local_Authority_Districts_December_2023_Boundaries_UK_BGC_2537431731774104276.GeoJSON"
)
england = lad[lad["LAD23CD"].str.startswith("E")]
print(f"Total LADs en Inglaterra: {len(england)}")

Total LADs en Inglaterra: 296
